In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.compose import make_column_transformer

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# -------------------------------
# Load + Feature Engineering SAME
# -------------------------------

df = pd.read_csv("Cleaned_Dataset.csv")

df["SalaryGrowth"] = df["PercentSalaryHike"] / (df["YearsAtCompany"] + 1)
df["PromotionDelay"] = df["YearsSinceLastPromotion"] / (df["YearsAtCompany"] + 1)
df["WorkPressure"] = df["OverTime"] * (4 - df["WorkLifeBalance"])
df["CareerStagnation"] = (df["YearsInCurrentRole"] + df["YearsSinceLastPromotion"]) / (df["TotalWorkingYears"] + 1)
df["Stability"] = df["YearsAtCompany"] / (df["TotalWorkingYears"] + 1)
df["IncomePerLevel"] = df["MonthlyIncome"] / (df["JobLevel"] + 1)
df["CommuteStress"] = df["DistanceFromHome"] * df["OverTime"]

df.drop([
    "PercentSalaryHike", "YearsAtCompany", "YearsSinceLastPromotion",
    "OverTime", "WorkLifeBalance", "YearsInCurrentRole",
    "TotalWorkingYears", "MonthlyIncome", "JobLevel", "DistanceFromHome"
], axis=1, inplace=True)

# -------------------------------
# Split
# -------------------------------
X = df.drop("Attrition", axis=1)
y = df["Attrition"]

cat_cols = ["BusinessTravel", "Department", "EducationField", "Gender", "JobRole", "MaritalStatus"]
num_cols = X.columns.difference(cat_cols)

col_trans = make_column_transformer(
    (OneHotEncoder(handle_unknown='ignore'), cat_cols),
    (StandardScaler(), num_cols)
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------------
# IMPROVED PIPELINE
# -------------------------------
pipe = Pipeline(steps=[
    ("preprocessor", col_trans),
    ("smote", SMOTE(random_state=42, sampling_strategy=0.6)),   # 🔥 NEW
    ("model", LogisticRegression(max_iter=5000, random_state=42))
])

# -------------------------------
# BETTER GRID
# -------------------------------
param_grid = {
    "model__C": np.logspace(-3, 2, 10),   # 🔥 smarter search
    "model__penalty": ["l1", "l2"],
    "model__solver": ["liblinear", "saga"],
    "model__class_weight": [None, "balanced"]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("\nBest Parameters:", grid.best_params_)

# -------------------------------
# Evaluation
# -------------------------------
y_prob = best_model.predict_proba(X_test)[:, 1]

# -------------------------------
# 🔥 WIDER THRESHOLD SEARCH
# -------------------------------
print("\n--- Threshold Tuning ---")

best_f1 = 0
best_threshold = 0

for t in np.arange(0.5, 0.7, 0.01):
    y_pred = (y_prob >= t).astype(int)

    f1 = f1_score(y_test, y_pred)

    print(f"Threshold: {t:.2f} | F1: {f1:.4f} | Recall: {recall_score(y_test, y_pred):.4f}")

    if f1 > best_f1 and recall_score(y_test, y_pred) > 0.55:
        best_f1 = f1
        best_threshold = t

print("\n🔥 Best Threshold:", best_threshold)

# -------------------------------
# Final Metrics
# -------------------------------
y_pred_final = (y_prob >= best_threshold).astype(int)

print("\n--- FINAL PRODUCTION METRICS ---")
print("Accuracy:", accuracy_score(y_test, y_pred_final))
print("Precision:", precision_score(y_test, y_pred_final))
print("Recall:", recall_score(y_test, y_pred_final))
print("F1 Score:", f1_score(y_test, y_pred_final))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

import joblib

# Save full pipeline
joblib.dump(best_model, "attrition_model.pkl")

print("✅ Model saved successfully")

import json

with open("threshold.json", "w") as f:
    json.dump({"threshold": float(best_threshold)}, f)

print("✅ Threshold saved")

with open("columns.json", "w") as f:
    json.dump(list(X.columns), f)

print("✅ Columns saved")

metrics = {
    "accuracy": accuracy_score(y_test, y_pred_final),
    "precision": precision_score(y_test, y_pred_final),
    "recall": recall_score(y_test, y_pred_final),
    "f1": f1_score(y_test, y_pred_final),
    "roc_auc": roc_auc_score(y_test, y_prob)
}

with open("metrics.json", "w") as f:
    json.dump(metrics, f)

print("✅ Metrics saved")

Fitting 5 folds for each of 80 candidates, totalling 400 fits

Best Parameters: {'model__C': 0.01291549665014884, 'model__class_weight': None, 'model__penalty': 'l2', 'model__solver': 'saga'}

--- Threshold Tuning ---
Threshold: 0.50 | F1: 0.5591 | Recall: 0.5532
Threshold: 0.51 | F1: 0.5495 | Recall: 0.5319
Threshold: 0.52 | F1: 0.5556 | Recall: 0.5319
Threshold: 0.53 | F1: 0.5287 | Recall: 0.4894
Threshold: 0.54 | F1: 0.5238 | Recall: 0.4681
Threshold: 0.55 | F1: 0.5238 | Recall: 0.4681
Threshold: 0.56 | F1: 0.5301 | Recall: 0.4681
Threshold: 0.57 | F1: 0.5301 | Recall: 0.4681
Threshold: 0.58 | F1: 0.5301 | Recall: 0.4681
Threshold: 0.59 | F1: 0.5432 | Recall: 0.4681
Threshold: 0.60 | F1: 0.5432 | Recall: 0.4681
Threshold: 0.61 | F1: 0.5570 | Recall: 0.4681
Threshold: 0.62 | F1: 0.5455 | Recall: 0.4468
Threshold: 0.63 | F1: 0.5263 | Recall: 0.4255
Threshold: 0.64 | F1: 0.5263 | Recall: 0.4255
Threshold: 0.65 | F1: 0.5067 | Recall: 0.4043
Threshold: 0.66 | F1: 0.4658 | Recall: 0.3617
